<div class="blog-language-switch" role="group" aria-label="Article language">
<span aria-current="page">English</span>
<a href="/ipynb/zh-CN/Computer-Science/Operating-Systems/03-threads-and-interprocess-communication.html" lang="zh-CN" hreflang="zh-CN">中文</a>
</div>

[Back to Operating Systems guideline](Operating-Systems.html)


## **Threads and Interprocess Communication**

A process gives a program an identity, an address space, and protected references to resources. A **thread** is an execution stream inside that process: it carries the register state and stack needed to run instructions, while reusing much of the process's memory and resource context. **Interprocess communication** (IPC) solves the complementary problem. It creates controlled paths between processes whose address spaces are intentionally isolated.

Those two choices move complexity to different places. Threads make data sharing cheap because ordinary loads and stores can reach common memory, but every thread must obey a synchronization protocol and one invalid pointer can damage the entire process. Separate processes improve isolation and failure containment, but communication must cross an explicit channel with rules for naming, copying or mapping, buffering, ordering, wakeups, and endpoint lifetime.

The chapter continues the pipeline introduced previously:

```bash
cat input.txt | grep kernel > result.txt
```

The real shell uses two processes and a pipe. This chapter asks what that design buys, how it would differ if `cat` and `grep` were threads in one process, and when an event-driven program might replace both blocking execution streams with explicit state machines. The focus is mechanism and interface semantics. CPU scheduling policy belongs to the next chapter, while locks, condition variables, atomicity, deadlock, and memory ordering are developed systematically in the synchronization chapter.

### **Why Do Programs Need Concurrent Execution?**

Programs often have several activities that can make progress independently: a server accepts new clients while older requests wait for storage, a user interface remains interactive while work runs in the background, and a pipeline lets a producer prepare later bytes while a consumer processes earlier bytes. Concurrent structure exposes that independence to a runtime or operating system.

Concurrency is not automatically faster. It introduces interleavings, communication, context switches, cache effects, queueing, and failure interactions. The reason to use it is that the problem contains independent work or unavoidable waiting and the resulting benefits exceed those costs.

#### **Concurrency and Parallelism**

**Concurrency** means that multiple activities have overlapping lifetimes and can make progress in an interleaved order. One CPU can support concurrency by switching among runnable threads. **Parallelism** means that multiple activities execute at the same instant on different processing resources. Parallelism therefore requires available hardware execution contexts, while concurrency is primarily a way to structure and schedule work.

![Two threads in one process interleave over time, illustrating concurrent progress even when only one thread executes at each instant.](assets/multithreaded-process.svg){fig-alt="Diagram of a process containing two threads whose execution intervals alternate down a time axis." width="56%"}

*Figure source: [Cburnett, Wikimedia Commons](https://commons.wikimedia.org/wiki/File:Multithreaded_process.svg), licensed under [CC BY-SA 3.0](https://creativecommons.org/licenses/by-sa/3.0/). The original SVG is stored locally for reliable rendering.*

The figure shows interleaving: thread 1 and thread 2 both advance over a time interval, but the CPU need not execute them simultaneously. On a multicore system, the scheduler can also place them on different CPUs and create genuine parallel execution.

Three distinctions prevent common misunderstandings:

- A concurrent program can be correct on one CPU and still be parallel on several CPUs. Its correctness must not depend on one particular timing.
- More threads do not create more independent work. If every task waits for one serial stage, additional threads mainly add queues and overhead.
- Hardware simultaneous multithreading, operating-system threads, and application tasks are different layers. A runtime may schedule thousands of tasks over tens of OS threads, which the kernel schedules over a smaller number of hardware contexts.

For CPU-bound work, the serial fraction limits speedup. If fraction `s` of a computation cannot be parallelized, an idealized upper bound with `N` workers is:

$$
S(N) = \frac{1}{s + \frac{1-s}{N}}
$$

If `s = 0.10`, infinitely many workers cannot exceed a speedup of `10`. Real performance is lower because task creation, communication, load imbalance, cache coherence, and memory bandwidth add cost. The equation is not a scheduler model; it is a reminder to locate the serial bottleneck before adding execution contexts.

#### **Responsiveness, Throughput, and Structure**

Concurrent execution can improve several different objectives:

| Objective | How concurrency helps | Typical failure if designed poorly |
|---|---|---|
| Responsiveness | interactive or control work can run while another activity waits | background work still holds a shared lock or saturates the CPU |
| Throughput | several requests overlap I/O and use otherwise idle resources | too much concurrency causes queueing, cache misses, or memory pressure |
| Latency hiding | one thread runs while another waits for storage, network, or a timer | unbounded queues hide overload until memory is exhausted |
| Parallel speedup | independent CPU work runs on several cores | contention and serial sections dominate |
| Program structure | activities map to comprehensible components or stages | shared mutable state creates timing-dependent coupling |
| Fault containment | separate worker processes can fail independently | a thread failure terminates or corrupts the whole process |

A web server illustrates the trade-off. A thread-per-request design allows simple blocking code, but a large number of mostly idle connections may consume stacks and kernel scheduling objects. An event loop can manage many idle descriptors with fewer threads, but every operation must avoid blocking and preserve explicit per-connection state. A process pool can contain crashes and apply different credentials, but communication and deployment become more explicit.

The correct design starts with workload questions: Is the work CPU-bound or waiting-bound? How many independent requests exist? Must failures be isolated? How large can queues grow? Does the protocol need message boundaries? Which resources can safely be shared? "Use more threads" is not an answer until those constraints are known.

### **The Thread Abstraction**

A thread is the smallest execution context an operating-system scheduler or language runtime can independently make runnable. It needs a program counter, registers, a stack, scheduling state, and a way to identify it. It does not normally own a separate process address space.

#### **Shared Process State and Per-Thread State**

Threads in one POSIX process execute the same program and share global memory, heap memory, memory mappings, open file descriptors, credentials, current directory, and signal dispositions. Each thread has its own registers, stacks, thread identifier, signal mask, scheduling attributes, `errno`, and thread-local values. Linux summarizes these responsibilities in [`pthreads(7)`](https://man7.org/linux/man-pages/man7/pthreads.7.html).

![Process-wide resources are shared by three threads, while registers, stacks, scheduler state, signal masks, errno, and TLS are private to each thread.](assets/thread-shared-private-state.svg){fig-alt="A process box containing shared address space, descriptors, identity, signal policy, resources, and lifetime, plus three thread boxes with private execution state." width="96%"}

*Figure: original teaching diagram for this chapter.*

The distinction is behavioral, not merely a memory-layout fact:

| Operation in one thread | Effect visible to peer threads |
|---|---|
| writes a global or heap object | peers can observe the same memory, subject to synchronization and memory-order rules |
| closes a file descriptor | the descriptor-table entry is gone for the process; another thread using that number may fail or hit a reused entry |
| changes the current working directory | pathname resolution changes for every thread in the process |
| changes its signal mask | only that thread's delivery eligibility changes |
| modifies a stack local | normally private because the stack is private, unless its address is shared |
| calls `exit()` or returns from `main()` | all threads terminate because the process exits |
| returns from its start routine or calls `pthread_exit()` | only that thread terminates |

"Stack local" does not guarantee isolation. A thread can pass the address of one of its local variables to another thread, and both can then access the same stack object. Correctness depends on object lifetime and synchronization, not on the variable's source-level spelling.

Sharing also changes failure behavior. A segmentation fault generated by one thread normally invokes the process-wide disposition and can terminate the entire process. Memory corruption may be even more subtle: the offending thread can overwrite a queue, allocator metadata, or another thread's data long before the process visibly fails. Threads are an execution isolation boundary, not a memory protection boundary.

#### **Thread Lifecycle and Thread Control Blocks**

A thread moves through lifecycle states similar to a process: it is created, becomes runnable, executes, blocks for an event, becomes runnable again, and terminates. The kernel or runtime records its resumable state in a **thread control block** (TCB) or equivalent structures.

A conceptual TCB contains:

- thread identity and lifecycle state;
- saved registers, stack pointer, and instruction pointer;
- references to user and kernel stacks;
- scheduling policy, priority, affinity, and accounting;
- per-thread signal mask and pending state;
- cancellation and cleanup state;
- thread-local storage metadata; and
- a reference to the process resources shared with peer threads.

In a one-to-one implementation, the kernel maintains scheduling state for every application thread. A user-level runtime may add another TCB for tasks that the kernel cannot see. These structures can coexist: a language task has runtime state, runs on an OS thread with kernel state, and belongs to a process that owns the address space.

Thread stacks deserve explicit resource planning. Reserving an 8 MiB virtual stack for each of 10,000 threads creates a large virtual-address commitment even if physical pages are allocated on demand. Making stacks too small risks overflow, especially with deep recursion or large automatic objects. Guard pages help convert some overflows into faults, but they do not choose a safe application stack size.

Termination and reclamation are separate events. A joinable thread can finish executing while retaining a small amount of status and bookkeeping until another thread joins it. A detached thread releases those final resources automatically. This resembles the reason a terminated process can remain waitable, although POSIX thread and process lifecycles use different APIs and kernel representations.

### **Thread Implementation Models**

The thread API seen by an application does not completely determine who schedules each execution stream. A runtime can switch user contexts itself, ask the kernel to schedule every thread, or combine both levels.

#### **User-Level Threads**

A **user-level thread** is represented and scheduled by a library or language runtime without requiring a kernel scheduling entity for every user task. Switching between two such threads can avoid a system call and can apply application-specific policies. Creating a task may require only a small stack and runtime metadata.

The kernel, however, schedules only the underlying kernel context. In a pure many-to-one design, if that context enters a blocking system call, the kernel cannot run another hidden user thread from the same process. The runtime must avoid blocking calls, use nonblocking I/O, or integrate with asynchronous facilities. The design also cannot run hidden threads in parallel on several CPUs because the kernel sees only one schedulable entity.

Modern runtimes often use the same idea without exposing it as POSIX threads. Fibers, coroutines, green threads, and async tasks can be multiplexed over an OS-thread pool. Their benefits depend on cooperation: tasks must yield at known points, and libraries must not unexpectedly block the carrier thread.

#### **Kernel Threads**

A **kernel-scheduled thread** has its own kernel execution and scheduling state. If one thread blocks in `read()`, another thread in the same process can remain runnable. The kernel can place threads from one process on several CPUs, preempt them independently, account for CPU usage, and honor per-thread affinity or real-time policy.

The price is kernel-visible state and transition cost. Creating and destroying a large number of threads allocates stacks and management structures. Scheduling too many runnable threads increases competition and cache disruption. Kernel support therefore enables parallelism and blocking APIs, but it does not make an unbounded thread-per-item design efficient.

Linux's modern Native POSIX Threads Library uses kernel facilities so each POSIX thread maps to a kernel scheduling entity. Linux still presents process-wide and thread-wide identities differently: tools can show a thread-group ID as the process PID and a distinct task ID for each thread.

#### **Many-to-One, One-to-One, and Many-to-Many Models**

![Many-to-one, one-to-one, and many-to-many mappings between user threads, runtime schedulers, kernel scheduling entities, and CPUs.](assets/thread-mapping-models.svg){fig-alt="Three-panel diagram comparing many user threads on one kernel thread, one user thread per kernel thread, and many user threads multiplexed over several kernel threads." width="96%"}

*Figure: original teaching diagram for this chapter.*

The models can be compared directly:

| Model | Scheduling relationship | Parallelism | Blocking behavior | Main implementation burden |
|---|---|---:|---|---|
| Many-to-one | runtime maps many user threads to one kernel entity | no | one blocking call can stall all user threads | runtime context switching and nonblocking integration |
| One-to-one | every user thread has a kernel entity | yes | one blocked thread does not inherently block peers | kernel objects and scheduling overhead per thread |
| Many-to-many | runtime maps many user threads over a kernel-thread pool | yes | runtime can continue work on available carriers | two-level scheduling, pinning, blocking, and migration rules |

Linux Pthreads uses a one-to-one model, as documented in [`pthreads(7)`](https://man7.org/linux/man-pages/man7/pthreads.7.html). A language runtime can still implement many-to-many scheduling above Pthreads by treating OS threads as carriers. Therefore, statements such as "this application has one million threads" are incomplete unless they identify whether those are language tasks, user fibers, POSIX threads, or kernel tasks.

### **Thread APIs and Lifecycle Management**

POSIX Pthreads gives C programs explicit control over thread creation, completion, attributes, cancellation, and thread-specific data. Most Pthread functions return an error number directly rather than returning `-1` and setting `errno`; checking them with the usual system-call pattern is incorrect.

#### **Creation, Joining, Detaching, and Cancellation**

`pthread_create()` starts a thread at a function that accepts and returns `void *`. After a successful call, either the creator or the new thread may run next. A joinable thread retains its final result until exactly one peer calls `pthread_join()`. A detached thread releases its remaining resources automatically and cannot later be joined. The Linux manual documents these contracts in [`pthread_create(3)`](https://man7.org/linux/man-pages/man3/pthread_create.3.html), [`pthread_join(3)`](https://man7.org/linux/man-pages/man3/pthread_join.3.html), and [`pthread_detach(3)`](https://man7.org/linux/man-pages/man3/pthread_detach.3.html).

<details>
<summary>Show an annotated Pthreads create-and-join example</summary>

```c
#define _POSIX_C_SOURCE 200809L
#include <pthread.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>

struct job {
    const char *text;       // Input remains valid until every join completes.
    size_t vowel_count;     // Each worker owns one distinct result slot.
};

static void report_pthread_error(const char *operation, int error) {
    // Pthread functions return the error number; they usually do not set errno.
    fprintf(stderr, "%s: %s\n", operation, strerror(error));
}

static void *count_vowels(void *argument) {
    struct job *job = argument;
    size_t count = 0;

    for (const char *p = job->text; *p != '\0'; ++p) {
        if (strchr("AEIOUaeiou", *p) != NULL) {
            ++count;
        }
    }

    // No two threads write the same field. pthread_join() establishes that
    // the worker has finished before main reads this result.
    job->vowel_count = count;
    return NULL;
}

int main(void) {
    struct job jobs[] = {
        {"process", 0},
        {"thread", 0},
        {"communication", 0}
    };
    enum { THREAD_COUNT = sizeof(jobs) / sizeof(jobs[0]) };
    pthread_t threads[THREAD_COUNT];
    size_t created = 0;

    for (; created < THREAD_COUNT; ++created) {
        int error = pthread_create(
            &threads[created], NULL, count_vowels, &jobs[created]
        );
        if (error != 0) {
            report_pthread_error("pthread_create", error);
            break;
        }
    }

    // Join every thread that was actually created, even after a later failure.
    for (size_t i = 0; i < created; ++i) {
        int error = pthread_join(threads[i], NULL);
        if (error != 0) {
            report_pthread_error("pthread_join", error);
            return EXIT_FAILURE;
        }
    }

    if (created != THREAD_COUNT) {
        return EXIT_FAILURE;
    }

    for (size_t i = 0; i < THREAD_COUNT; ++i) {
        printf("%s -> %zu vowels\n", jobs[i].text, jobs[i].vowel_count);
    }
    return EXIT_SUCCESS;
}
```

```bash
cc -std=c11 -Wall -Wextra -O2 -pthread threads.c -o threads
./threads
```

</details>

The example avoids a shared counter. Each worker writes a distinct object and the creator reads results only after joining. Replacing the result array with one unsynchronized global counter would introduce a data race; `counter++` is a read-modify-write sequence, not an indivisible mathematical operation.

Cancellation is more difficult than termination. `pthread_cancel()` sends a request; default deferred cancellation acts at defined cancellation points. A thread canceled while holding a resource can leave invariants broken unless cleanup handlers release what it owns. Asynchronous cancellation can interrupt almost any instruction and is safe for very little code. Structured stop flags, closed queues, deadlines, or explicit wakeup descriptors often make shutdown easier to reason about than arbitrary cancellation.

| Lifecycle choice | Result collection | Final resource release | Suitable use |
|---|---|---|---|
| Joinable | one peer joins and can obtain the returned value | after successful join | finite tasks whose completion matters |
| Detached | no join and no result collection | automatically at termination | independent background work with separately managed shutdown |
| Canceled | depends on joinable/detached state | after cancellation cleanup and join or automatic detach cleanup | cooperative interruption where all owned resources are accounted for |

A program should not simply detach every thread to avoid joins. Detached work still accesses process memory; exiting `main`, unloading a library, or freeing shared state while it runs can create use-after-free failures.

#### **Thread-Local Storage**

**Thread-local storage** (TLS) gives each thread a distinct instance of a variable while preserving a common source-level name or key. It is useful for `errno`, per-thread caches, random-number state, logging context, and library data that would otherwise require passing a context parameter through every call.

![Thread-local storage keys resolve to different values and local objects in thread 1 and thread 2.](assets/tls-principle.png){fig-alt="A process contains global TLS keys that index separate TLS arrays and local objects for two threads." width="72%"}

*Figure source: [Mac LAK, Wikimedia Commons](https://commons.wikimedia.org/wiki/File:TLS_principle.svg), licensed under [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/). A local rasterization of the original SVG is used for consistent rendering.*

C provides language-level thread storage duration through `_Thread_local`; compilers and loaders arrange one instance per thread. Pthreads also supplies dynamic keys through `pthread_key_create()`, `pthread_setspecific()`, and `pthread_getspecific()`. A key can have a destructor that is invoked for non-null thread-specific values when a thread exits, subject to the API's repeated-destructor rules.

<details>
<summary>Show static and dynamic TLS in C</summary>

```c
#include <pthread.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>

static _Thread_local unsigned long local_requests = 0;
static pthread_key_t label_key;

static void destroy_label(void *value) {
    free(value);
}

static void *worker(void *argument) {
    const char *input_label = argument;
    char *private_label = strdup(input_label);
    if (private_label == NULL) {
        return NULL;
    }

    int error = pthread_setspecific(label_key, private_label);
    if (error != 0) {
        free(private_label);
        return NULL;
    }

    for (int i = 0; i < 3; ++i) {
        ++local_requests; // This thread updates only its own TLS instance.
    }

    printf("%s handled %lu requests\n",
           (const char *)pthread_getspecific(label_key), local_requests);
    return NULL; // destroy_label releases the dynamic TLS value.
}

int main(void) {
    if (pthread_key_create(&label_key, destroy_label) != 0) {
        return EXIT_FAILURE;
    }

    pthread_t first, second;
    int error = pthread_create(&first, NULL, worker, "worker-A");
    if (error != 0) {
        fprintf(stderr, "pthread_create: %s\n", strerror(error));
        return EXIT_FAILURE;
    }

    error = pthread_create(&second, NULL, worker, "worker-B");
    if (error != 0) {
        fprintf(stderr, "pthread_create: %s\n", strerror(error));
        pthread_join(first, NULL);
        pthread_key_delete(label_key);
        return EXIT_FAILURE;
    }

    pthread_join(first, NULL);
    pthread_join(second, NULL);
    pthread_key_delete(label_key);
    return EXIT_SUCCESS;
}
```

</details>

TLS makes the pointer slot private, not necessarily the object it points to. If every thread stores a pointer to the same heap object in TLS, that heap object is still shared and still requires a protocol. Excessive TLS can also hide dependencies and multiply memory usage by the thread count.

### **Why Interprocess Communication Is Necessary**

Separate processes normally have separate virtual address spaces and descriptor tables. A pointer meaningful in process A does not grant process B access to the same virtual address or physical memory. That isolation prevents accidental writes from becoming ordinary cross-process corruption, but cooperating programs need kernel-mediated ways to discover endpoints, transfer data, share selected memory, and wait for state changes.

IPC is therefore not one feature. It is a design space that includes byte streams, discrete messages, shared mappings, notifications, and local or network-style sockets. Every mechanism defines both a data plane and a control contract.

#### **Naming, Buffering, Ordering, and Backpressure**

Before selecting an IPC API, specify its semantics:

| Question | Examples of possible answers | Why it matters |
|---|---|---|
| How are peers named? | inherited descriptor, filesystem path, POSIX name, socket address, passed handle | determines discovery, permissions, and lifetime |
| Where is data buffered? | sender memory, kernel buffer, receiver memory, shared region | determines capacity, copying, and failure behavior |
| Are records preserved? | byte stream, fixed-size records, datagrams, sequenced packets | determines framing and partial-operation logic |
| What ordering exists? | per-stream byte order, message priority, no cross-sender total order | determines what receivers may infer |
| What happens at capacity? | sender blocks, operation returns `EAGAIN`, data is rejected or overwritten | defines backpressure and overload behavior |
| How is closure observed? | EOF, hangup readiness, zero-length receive, explicit sentinel | defines graceful shutdown |
| How is peer authority checked? | inherited trust, filesystem permissions, credentials, labels | defines security assumptions |
| What survives a crash? | nothing, named kernel object, buffered messages, persistent file | defines recovery and cleanup |

Backpressure is especially important. A queue that can grow without a limit converts temporary overload into delayed memory exhaustion. A bounded queue forces a policy: block the producer, reject work, shed low-priority data, or propagate slower demand upstream. The choice is part of application correctness, not merely performance tuning.

IPC calls can also complete partially. A stream `read()` may return fewer bytes than requested, a nonblocking send may accept only a prefix, and readiness can disappear before a competing thread performs the operation. Robust protocols treat return values and retry state as first-class data.

#### **Copying Versus Shared State**

Message-oriented IPC normally makes transfer explicit: a sender provides bytes, the kernel validates an endpoint and buffering state, and a receiver later obtains bytes. Shared memory instead maps selected physical storage into more than one address space so programs communicate through loads and stores.

![Comparison between kernel-mediated message transfer and shared-memory mappings, including their different protocol responsibilities.](assets/ipc-copy-versus-sharing.svg){fig-alt="Two-panel diagram comparing user buffers connected through a kernel channel with two process mappings connected to the same physical shared frames." width="96%"}

*Figure: original teaching diagram for this chapter.*

The trade-off is not simply "copying is slow, shared memory is fast."

- Kernel channels provide explicit operations, endpoint lifetime, blocking rules, and natural places to enforce permissions and accounting.
- Shared memory can avoid repeated payload copies, but applications must define object layout, ownership, publication, synchronization, wakeups, and recovery from a peer dying mid-update.
- Small messages often spend more time in system-call, scheduling, cache, and protocol work than in the byte copy itself.
- Some APIs use optimized page sharing, scatter/gather I/O, or kernel buffers, so the phrase "zero copy" must identify which copy and which ownership transition disappeared.
- Shared frames still participate in cache coherence. Two cores repeatedly writing the same cache line can make a shared-memory design slower than a message design with clear ownership.

Choose the mechanism from semantics first. Optimize transfer only after measuring payload size, message rate, cache behavior, wakeups, and contention.

### **Pipes and FIFOs**

An anonymous **pipe** is a kernel byte-stream object accessed through a read descriptor and a write descriptor. Processes usually obtain access by inheritance after `fork()` or by receiving a descriptor from another process. A **FIFO**, also called a named pipe, gives a pipe-like channel a filesystem name so unrelated processes can open it.

Pipes are streams, not message queues. If a writer performs two writes, a reader may receive both in one read or split one write across several reads. POSIX only guarantees that sufficiently small writes of at most `PIPE_BUF` bytes are not interleaved with data from competing writers. It does not make those writes independently discoverable records.

![Animated pipe model showing a writer, bounded kernel byte buffer, reader, empty and full blocking states, and EOF after the last writer closes.](assets/pipe-backpressure-animated.svg){fig-alt="Animated diagram of bytes moving from a writer through a bounded kernel pipe buffer to a reader, with four static panels explaining empty, flowing, full, and EOF states." width="96%"}

*Figure: original teaching animation for this chapter. The complete semantics remain visible when animation is unavailable.*

The official Linux [`pipe(7)`](https://man7.org/linux/man-pages/man7/pipe.7.html) page documents the key behavior:

- A blocking read waits when the pipe is empty while at least one writer remains.
- Once all write descriptors are closed, a reader drains buffered bytes and then receives `0`, the stream EOF indication.
- A blocking write waits when the bounded buffer lacks capacity. A nonblocking operation returns `-1` with `EAGAIN` instead.
- Writing when no reader exists generates `SIGPIPE`; if that signal is ignored or handled, the write fails with `EPIPE`.
- Writes no larger than `PIPE_BUF` have an inter-writer atomicity guarantee, subject to blocking mode. Larger writes can interleave.

These rules explain a pipeline's flow control. If `grep` consumes slowly, the pipe fills and `cat` eventually blocks. The kernel does not need an application-specific speed controller; bounded capacity and blocking propagate pressure upstream.

FIFOs add naming but do not turn the stream into persistent storage. Opening semantics can block until the opposite endpoint exists, and the FIFO's directory permissions influence who can open it. The filesystem entry identifies the channel; bytes still live in a kernel pipe object and disappear when no relevant open endpoints remain.

<details>
<summary>Show a FIFO observation workflow</summary>

```bash
fifo="/tmp/os03-fifo-$$"
mkfifo -m 600 "$fifo"

# The reader opens first and may wait until a writer opens the FIFO.
grep kernel < "$fifo" > result.txt &
reader_pid=$!

# The writer's stdout becomes the FIFO byte stream.
cat input.txt > "$fifo"

wait "$reader_pid"
rm -f "$fifo"
cat result.txt
```

Run `strace -f -e trace=openat,read,write,close,wait4` around a small shell script containing these commands to observe endpoint creation, blocking, transfer, EOF, and collection.

</details>

An application protocol layered on a pipe must define framing. Common choices include newline-delimited text, a fixed-size record, a length prefix followed by exact bytes, or an escaping scheme. It must also place an upper bound on accepted lengths before allocating memory; trusting an arbitrary length prefix turns framing into a memory-exhaustion vulnerability.

### **Signals and Asynchronous Notifications**

A **signal** is a small asynchronous notification associated with an event such as terminal input, a timer, an invalid memory access, child state change, or an explicit request from another process. It is well suited to control events, but it is not a general data transport. Standard signals mainly carry identity and limited metadata, and multiple pending instances of the same standard signal may collapse into one.

![Signal flow from generation through pending state and thread-mask selection to a default action, asynchronous handler, synchronous sigwait call, or signalfd.](assets/signal-delivery-path.svg){fig-alt="Flowchart showing signal generation, pending sets, selection of an eligible thread, and asynchronous or synchronous consumption paths." width="96%"}

*Figure: original teaching diagram for this chapter.*

Signal state has both process-wide and per-thread parts:

- A process shares dispositions that say whether a signal uses the default action, is ignored, or invokes a handler.
- Each thread has its own signal mask. A blocked signal remains pending rather than being erased.
- A signal can be process-directed, allowing delivery to an eligible thread, or thread-directed through interfaces such as `pthread_kill()` or as a consequence of a thread-specific hardware fault.
- Standard and real-time signals have different queueing rules; real-time signals can carry values and are queued in order within their defined priority rules.

An asynchronous handler interrupts ordinary control flow. The kernel saves context in a signal frame, arranges execution at the handler, and later restores the interrupted context through a signal-return path. Almost all library functions are unsafe there because the interrupted thread may already hold internal library state. A handler should normally set a `sig_atomic_t` flag, write to a prearranged descriptor using an async-signal-safe operation, or perform another deliberately minimal action. Linux describes the delivery sequence and safe constraints in [`signal(7)`](https://man7.org/linux/man-pages/man7/signal.7.html).

Multithreaded servers often use a synchronous design instead: block selected signals before creating workers so they inherit the mask, then dedicate one thread to `sigwait()`. The signal thread receives notifications through normal function return and can use ordinary thread-safe program structure afterward.

<details>
<summary>Show a dedicated sigwait thread with cooperative shutdown</summary>

```c
#define _POSIX_C_SOURCE 200809L
#include <pthread.h>
#include <signal.h>
#include <stdatomic.h>
#include <stdbool.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <time.h>

static atomic_bool stop_requested = false;
static sigset_t termination_signals;

static void *signal_thread(void *unused) {
    (void)unused;
    int signal_number;
    int error = sigwait(&termination_signals, &signal_number);
    if (error != 0) {
        fprintf(stderr, "sigwait: %s\n", strerror(error));
        atomic_store(&stop_requested, true);
        return NULL;
    }

    printf("accepted signal %d in ordinary thread context\n", signal_number);
    atomic_store(&stop_requested, true);
    return NULL;
}

static void *worker(void *argument) {
    const char *name = argument;
    struct timespec delay = {.tv_sec = 0, .tv_nsec = 200000000};

    while (!atomic_load(&stop_requested)) {
        printf("%s completed one unit\n", name);
        nanosleep(&delay, NULL);
    }
    printf("%s stopping cleanly\n", name);
    return NULL;
}

int main(void) {
    sigemptyset(&termination_signals);
    sigaddset(&termination_signals, SIGINT);
    sigaddset(&termination_signals, SIGTERM);

    // Block before creating threads; every new thread inherits this mask.
    int error = pthread_sigmask(SIG_BLOCK, &termination_signals, NULL);
    if (error != 0) {
        fprintf(stderr, "pthread_sigmask: %s\n", strerror(error));
        return EXIT_FAILURE;
    }

    pthread_t controller, first, second;
    error = pthread_create(&first, NULL, worker, "worker-A");
    if (error != 0) {
        fprintf(stderr, "pthread_create worker-A: %s\n", strerror(error));
        return EXIT_FAILURE;
    }

    error = pthread_create(&second, NULL, worker, "worker-B");
    if (error != 0) {
        fprintf(stderr, "pthread_create worker-B: %s\n", strerror(error));
        atomic_store(&stop_requested, true);
        pthread_join(first, NULL);
        return EXIT_FAILURE;
    }

    error = pthread_create(&controller, NULL, signal_thread, NULL);
    if (error != 0) {
        fprintf(stderr, "pthread_create signal thread: %s\n", strerror(error));
        atomic_store(&stop_requested, true);
        pthread_join(first, NULL);
        pthread_join(second, NULL);
        return EXIT_FAILURE;
    }

    printf("press Ctrl-C to request shutdown\n");
    pthread_join(controller, NULL);
    pthread_join(first, NULL);
    pthread_join(second, NULL);
    return EXIT_SUCCESS;
}
```

</details>

The C11 atomic flag makes the cross-thread stop request defined. The synchronization chapter will explain why an ordinary `bool` would create a data race and how memory-order choices affect visibility. On Linux, [`signalfd(2)`](https://man7.org/linux/man-pages/man2/signalfd.2.html) can expose blocked signals as descriptor records, allowing one event loop to handle signals alongside pipes, sockets, and timers.

### **Shared Memory and Memory-Mapped Communication**

Shared memory maps the same backing storage into more than one process. POSIX named shared memory uses `shm_open()` to obtain a descriptor, `ftruncate()` to set object size, and `mmap(..., MAP_SHARED, ...)` to create mappings. Linux summarizes that lifecycle in [`shm_overview(7)`](https://man7.org/linux/man-pages/man7/shm_overview.7.html).

![POSIX shared-memory lifecycle from naming and sizing through mapping, protocol design, unmapping, descriptor closure, and unlinking.](assets/shared-memory-lifecycle.svg){fig-alt="Four-stage diagram showing shm_open and ftruncate, two process mappings to shared frames, data and synchronization protocol, and munmap close shm_unlink cleanup." width="96%"}

*Figure: original teaching diagram for this chapter.*

Several distinct objects and lifetimes are involved:

1. The **name** lets processes discover the shared-memory object and is governed by ownership and permissions.
2. The **descriptor** refers to the object and can be closed after mapping if no descriptor operation is needed.
3. The **mapping** places the object into one process's virtual address space. Different processes can map it at different virtual addresses.
4. The **backing storage** remains while names or live references keep it reachable according to platform rules.

Because virtual addresses can differ, a shared region should not contain ordinary absolute pointers that another process will dereference. Store offsets, indices, fixed-width values, or relocatable structures. A robust layout also contains a magic value, version, total size, and bounds that every participant validates before trusting payload data.

`MAP_SHARED` permits updates to become visible, but it does not define when a multi-field record becomes complete. Concurrent participants need process-shared mutexes, semaphores, futex-based protocols, or language/ABI-supported atomics with a correct memory-order design. They also need a wakeup strategy; repeatedly polling a flag can waste CPU even if the flag itself is atomic.

<details>
<summary>Show a minimal POSIX shared-memory example</summary>

```c
#define _POSIX_C_SOURCE 200809L
#include <fcntl.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <sys/mman.h>
#include <sys/stat.h>
#include <sys/types.h>
#include <sys/wait.h>
#include <unistd.h>

struct shared_record {
    size_t length;
    char text[256];
};

int main(void) {
    char name[64];
    snprintf(name, sizeof(name), "/os03-shm-%ld", (long)getpid());

    int fd = shm_open(name, O_CREAT | O_EXCL | O_RDWR, 0600);
    if (fd == -1) {
        perror("shm_open");
        return EXIT_FAILURE;
    }
    if (ftruncate(fd, sizeof(struct shared_record)) == -1) {
        perror("ftruncate");
        shm_unlink(name);
        close(fd);
        return EXIT_FAILURE;
    }

    struct shared_record *record = mmap(
        NULL, sizeof(*record), PROT_READ | PROT_WRITE, MAP_SHARED, fd, 0
    );
    if (record == MAP_FAILED) {
        perror("mmap");
        shm_unlink(name);
        close(fd);
        return EXIT_FAILURE;
    }
    close(fd); // The mapping now retains the live reference we need.

    pid_t child = fork();
    if (child == -1) {
        perror("fork");
        munmap(record, sizeof(*record));
        shm_unlink(name);
        return EXIT_FAILURE;
    }

    if (child == 0) {
        const char *message = "shared pages, separate processes";
        record->length = (size_t)snprintf(
            record->text, sizeof(record->text), "%s", message
        );
        munmap(record, sizeof(*record));
        _exit(0);
    }

    // waitpid provides the sequencing for this one-write demonstration.
    // A long-lived concurrent channel needs its own synchronization protocol.
    if (waitpid(child, NULL, 0) == -1) {
        perror("waitpid");
    } else if (record->length < sizeof(record->text)) {
        printf("received: %.*s\n", (int)record->length, record->text);
    }

    munmap(record, sizeof(*record));
    shm_unlink(name); // Remove the discovery name after the participants finish.
    return EXIT_SUCCESS;
}
```

```bash
cc -std=c11 -Wall -Wextra -O2 shared_memory.c -o shared_memory -lrt
./shared_memory
```

</details>

The example uses `waitpid()` as a simple completion barrier, so parent and child do not access a partially published record concurrently. It demonstrates mapping and lifetime, not a high-throughput channel. A production shared ring buffer must define producer and consumer indices, full and empty states, memory visibility, peer death, and what happens when a producer stalls after reserving a slot.

### **Message Queues and Local Sockets**

Message queues preserve discrete records instead of exposing one undifferentiated byte stream. POSIX message queues are named kernel objects opened with `mq_open()`. They have bounded message counts and sizes, can attach priorities, and can block or return `EAGAIN` when full or empty. Their names and messages can outlive the process that created them until the queue is unlinked, subject to implementation limits. Linux documents the interface in [`mq_overview(7)`](https://man7.org/linux/man-pages/man7/mq_overview.7.html).

**Unix-domain sockets** provide local bidirectional communication through the socket API. They support stream, datagram, and sequenced-packet semantics. A pathname can name a listening endpoint; `socketpair()` creates two connected unnamed endpoints particularly useful between related processes. On supporting systems, ancillary messages can pass file descriptors and peer credentials, turning a socket into both a data channel and a controlled authority-transfer path.

| Mechanism | Data model | Connection shape | Useful properties | Main cautions |
|---|---|---|---|---|
| POSIX message queue | bounded discrete messages with priorities | many processes open one named queue | explicit records, asynchronous send/receive, notification | system limits, name cleanup, priority policy |
| Unix `SOCK_STREAM` | ordered byte stream | connected bidirectional peers | familiar socket API, backpressure, local credentials | application framing and partial I/O required |
| Unix `SOCK_DGRAM` | discrete datagrams | connectionless or connected local endpoints | message boundaries, one send per datagram | size limits, delivery/error semantics vary by platform |
| Unix `SOCK_SEQPACKET` | ordered reliable records | connected bidirectional peers | preserves message boundaries with connection semantics | less universally available than stream sockets |
| `socketpair()` | type selected by caller | one preconnected pair | no pathname race; natural after fork | only directly creates a pair, not a multi-client listener |

<details>
<summary>Show a local SOCK_SEQPACKET request-response example</summary>

```c
#define _GNU_SOURCE
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <sys/socket.h>
#include <sys/types.h>
#include <sys/wait.h>
#include <unistd.h>

int main(void) {
    int sockets[2];
    if (socketpair(
            AF_UNIX, SOCK_SEQPACKET | SOCK_CLOEXEC, 0, sockets
        ) == -1) {
        perror("socketpair");
        return EXIT_FAILURE;
    }

    pid_t child = fork();
    if (child == -1) {
        perror("fork");
        return EXIT_FAILURE;
    }

    if (child == 0) {
        close(sockets[0]);
        char request[128];
        ssize_t received = recv(sockets[1], request, sizeof(request) - 1, 0);
        if (received <= 0) {
            _exit(1);
        }
        request[received] = '\0';
        dprintf(STDOUT_FILENO, "child received one record: %s\n", request);

        const char reply[] = "ACK";
        if (send(sockets[1], reply, sizeof(reply) - 1, 0) == -1) {
            _exit(1);
        }
        close(sockets[1]);
        _exit(0);
    }

    close(sockets[1]);
    const char request[] = "filter=kernel; file=input.txt";
    if (send(sockets[0], request, sizeof(request) - 1, 0) == -1) {
        perror("send");
    }

    char reply[32];
    ssize_t received = recv(sockets[0], reply, sizeof(reply) - 1, 0);
    if (received > 0) {
        reply[received] = '\0';
        printf("parent received one record: %s\n", reply);
    }

    close(sockets[0]);
    waitpid(child, NULL, 0);
    return EXIT_SUCCESS;
}
```

</details>

Unlike a stream, `SOCK_SEQPACKET` preserves record boundaries, but a receive buffer can still be too small for a record and truncation must be handled according to the API. Linux details address forms, permissions, credentials, and ancillary data in [`unix(7)`](https://man7.org/linux/man-pages/man7/unix.7.html). Passing a descriptor with `SCM_RIGHTS` does not send the sender's integer descriptor number; it installs a new descriptor referring to the same underlying open-file description in the receiver.

### **Blocking, Nonblocking, and Event-Driven Communication**

A **blocking** operation allows the calling thread to sleep until progress is possible. This produces straightforward sequential code and lets the kernel avoid busy waiting. It consumes an execution context while the operation is logically pending, although a sleeping thread does not consume a CPU.

A **nonblocking** descriptor changes "this operation would have to sleep" into an immediate result such as `EAGAIN`. The application must retain partial progress and try again when the endpoint becomes ready. Repeatedly retrying in a tight loop is still busy waiting; scalable designs combine nonblocking I/O with a readiness interface such as `select()`, `poll()`, Linux `epoll`, or BSD `kqueue`.

![Readiness-driven event loop registering pipes, sockets, timers, and signal descriptors, then waiting, receiving a batch, draining I/O, and updating state.](assets/event-driven-ipc-loop.svg){fig-alt="Flow diagram showing several descriptor event sources feeding a readiness monitor and a four-step event loop that waits, receives ready descriptors, performs I/O, and updates state." width="96%"}

*Figure: original teaching diagram for this chapter.*

Readiness means that an operation can make progress without blocking **at the time of the notification**. It does not promise that a full logical message is available, that a large write will complete, or that another thread has not consumed the data first. Code must still inspect every operation's return value.

Level-triggered readiness continues to report an endpoint while the condition remains true. Edge-triggered readiness reports a transition and usually requires draining the nonblocking endpoint until `EAGAIN`; stopping early can leave unread data without another edge. Linux's [`epoll(7)`](https://man7.org/linux/man-pages/man7/epoll.7.html) explicitly recommends this drain-and-record-state pattern for edge-triggered loops.

<details>
<summary>Show event-loop pseudocode with partial-operation state</summary>

```text
set every managed endpoint to nonblocking mode
create readiness instance
register desired read, write, close, and error conditions

while service is running:
    ready_batch = wait_for_events(deadline)

    for event in ready_batch:
        connection = lookup_state(event.descriptor)

        if event says readable:
            repeat:
                result = read_into_available_input_space(connection)
                if result > 0:
                    parse_every_complete_message(connection)
                else if result == 0:
                    mark_peer_end_of_stream(connection)
                    break
                else if error is EAGAIN:
                    break
                else:
                    fail_connection(connection)
                    break

        if event says writable:
            repeat while output remains:
                result = write_unsent_suffix(connection)
                if result > 0:
                    advance_output_offset(connection, result)
                else if error is EAGAIN:
                    break
                else:
                    fail_connection(connection)
                    break

        update_interest_set(connection)
        close_only_when_protocol_and_buffer_state_allow_it
```

</details>

An event loop reduces the number of blocked stacks and scheduler entities, but it does not eliminate concurrency. Events can arrive in any order, handlers share mutable state, and work can be offloaded to thread pools. A slow CPU-heavy callback blocks progress for every endpoint on that loop. Hybrid designs therefore use event-driven I/O with a bounded worker pool and an explicit completion channel.

| Design | Programming model | Strength | Typical limit |
|---|---|---|---|
| one blocking thread per flow | sequential operations per thread | simple local reasoning and broad library compatibility | stack and scheduling overhead at very high flow counts |
| one event loop | explicit state machine and nonblocking operations | many mostly idle endpoints with few threads | callback latency, partial-state complexity, accidental blocking |
| event loop plus worker pool | readiness front end with bounded CPU tasks | separates I/O scale from CPU parallelism | queue sizing, ownership transfer, and result routing |

### **Connecting the Pipeline with Processes, Threads, and IPC**

The shell pipeline is one point in a larger design space. Its process boundary means `cat` and `grep` can be independently developed, executed, credentialed, traced, and terminated. The pipe supplies ordered bytes, bounded buffering, blocking wakeups, and EOF without either program knowing the other's implementation.

![The same input-filter-output workflow implemented as isolated processes and a pipe, threads and a shared queue, or an event loop and explicit state machines.](assets/pipeline-concurrency-designs.svg){fig-alt="Three-row comparison of a process pipeline, a threaded pipeline with a shared bounded queue, and a single-threaded event-loop design." width="96%"}

*Figure: original teaching diagram for this chapter.*

Rewriting the workflow with threads changes several obligations:

- A shared queue replaces the kernel pipe if the goal is to avoid copying payload between address spaces.
- The program must implement queue capacity, ownership, wakeups, end-of-input, cancellation, and error propagation.
- `cat` and `grep` are no longer ordinary executable components; they become functions or libraries compatible with one process.
- A memory fault or corrupt shared object can destroy both stages.
- The implementation may avoid process startup and some copying, but synchronization and cache coherence can offset those savings.

An event-loop version removes the dedicated blocking execution stream for each stage. It retains explicit state for input, filtering, and output and advances whichever operation is ready. That can be effective when many independent pipelines are mostly waiting, but the text filtering itself must remain short or move to workers.

The original process pipeline is therefore not an outdated substitute for threads. It is a composition architecture with strong isolation and a universal byte-stream interface. Threads are preferable when stages intentionally share a rich in-memory model and trust one another. Event-driven designs are preferable when the dominant problem is managing many asynchronous endpoints with bounded execution contexts.

<details>
<summary>Show Linux commands for observing process and thread boundaries</summary>

```bash
# Process hierarchy and process-level state.
ps -eo pid,ppid,pgid,stat,cmd --forest

# Threads are listed as lightweight processes (LWPs) within each process.
ps -eLo pid,tid,ppid,psr,stat,comm

# Inspect every thread task directory for one process.
ls "/proc/<PID>/task"
for task in /proc/<PID>/task/*; do
    grep -E '^(Name|Pid|Tgid|State|voluntary|nonvoluntary)' "$task/status"
done

# Follow process creation, descriptors, signals, and IPC-related calls.
strace -ff -e trace=process,desc,signal,ipc -o os03.trace \
  sh -c 'cat input.txt | grep kernel > result.txt'

# Report per-thread CPU activity for a live multithreaded process.
pidstat -t -p <PID> 1
```

</details>

On Linux, `Tgid` identifies the thread group commonly presented as the process PID, while `Pid` in a task status file identifies that kernel task. `pthread_t`, Linux task IDs, and process IDs are related through an implementation but are not interchangeable API types.

### **Comparison and Summary**

No IPC mechanism dominates every workload. The following matrix captures the primary semantics rather than implementation-specific benchmark numbers:

| Mechanism | Payload model | Natural peer relationship | Backpressure / waiting | Isolation and protocol burden |
|---|---|---|---|---|
| anonymous pipe | ordered byte stream | usually related processes sharing descriptors | bounded kernel buffer; block or `EAGAIN` | strong address-space isolation; application framing |
| FIFO | ordered byte stream with a filesystem name | unrelated local processes | pipe semantics plus open-time coordination | pathname permissions and cleanup |
| signal | notification plus limited metadata | process or thread targets | pending/mask rules, not bulk-data flow control | asynchronous safety or dedicated acceptance thread |
| shared memory | application-defined memory layout | processes that map the same object | application-defined synchronization and wakeups | lowest transfer boundary, highest shared-state burden |
| POSIX message queue | bounded discrete prioritized messages | processes opening one name | block, timed wait, or `EAGAIN` | kernel-maintained records and named-object cleanup |
| Unix stream socket | ordered bidirectional byte streams | local clients and servers or socket pairs | socket buffers and readiness | framing required; credentials and descriptor passing available |
| Unix sequenced-packet socket | ordered bidirectional records | connected local peers | message-aware send/receive and readiness | preserves records; platform portability must be checked |

A practical selection sequence is:

1. Choose the protection boundary. If a memory bug must not corrupt a peer, start with processes rather than threads.
2. Choose the data model. Decide whether the protocol is a stream, discrete records, shared objects, or notifications.
3. Bound every queue and define overload behavior.
4. Define endpoint discovery, permissions, lifetime, closure, and peer failure.
5. Decide whether blocking threads, an event loop, or a hybrid best matches the expected concurrency.
6. Measure copying, wakeups, contention, cache behavior, and memory use only after semantics are correct.

Common misconceptions can now be corrected precisely:

- **"Concurrency means simultaneous execution."** Concurrency permits interleaving; parallelism is simultaneous execution.
- **"Threads have separate memory."** They have separate execution state and stacks but normally share the process address space and descriptors.
- **"A local variable is always thread-private."** Its stack storage is private until an address or reference is shared.
- **"TLS makes an object private."** TLS makes each slot private; slots can still point to one shared object.
- **"Pipes preserve writes as messages."** Pipes preserve byte order and limited inter-writer atomicity, not record boundaries.
- **"Signals are a queue of events."** Standard signals may coalesce, and asynchronous handlers have severe safety restrictions.
- **"Shared memory requires no IPC overhead."** It removes an explicit payload-transfer path but adds layout, synchronization, visibility, wakeup, and recovery responsibilities.
- **"Nonblocking means the operation will succeed immediately."** It means the call will not sleep; `EAGAIN` and partial progress are normal.
- **"Readiness means one complete message is available."** It only means an operation can currently make some progress.

The pipeline now has a complete concurrency explanation. Processes provide isolation, threads provide shared execution within one process, and IPC defines controlled cooperation across protection boundaries. The next chapter, [CPU Scheduling and Dispatch](04-cpu-scheduling-and-dispatch.html), explains how runnable threads compete for processors. [Synchronization, Concurrent Correctness, and Liveness](05-synchronization-correctness-and-liveness.html) then develops the locks, conditions, atomics, memory-order rules, and deadlock reasoning needed by the shared-state designs introduced here.
